# Fine-tuning on Hand-Annotated Data

Fine-tunes the cross-validated MobileNetV3-UNet checkpoint (from
`01_cv_baseline_and_augmentation.ipynb`) on a small (~190-image)
hand-annotated set combining synthetic renders, real robot-camera frames,
and stock photos.

**Why fine-tune at all:** the cross-validated model showed a specific
failure mode in deployment testing — labeling parts of the robot's own
chassis/nearby obstacles as floor (a false-positive problem). This
fine-tune targets that directly.

**Loss:** an ignore-mask-aware, asymmetric Tversky+BCE loss
(`FP_WEIGHT=0.80`) that penalizes false positives more heavily than false
negatives, rather than the symmetric loss used in cross-validation.

**Schedule:** only the last encoder stage plus the full decoder are
trainable (`PHASE1_EPOCHS=20, PHASE2_EPOCHS=0`) — the rest of the
ImageNet-pretrained encoder stays frozen exactly as the cross-validated
checkpoint left it, since fine-tuning the full network on ~190 images
risks overwriting what cross-validation already learned.

**Precision:** runs in FP32 (`USE_AMP=False`) — mixed-precision (fp16
autocast) was found to produce non-finite loss on every batch on the
development GPU; disabling it resolved this completely. The `assert`
statements throughout this notebook checking `USE_AMP is False` and
`not torch.is_autocast_enabled()` are a direct result of that debugging
process, kept as guardrails against the same class of failure recurring.

**Result** (same held-out validation split, source checkpoint vs.
fine-tuned):

| | IoU | Precision | Recall |
|---|---|---|---|
| Source checkpoint | 0.859 | 0.967 | 0.885 |
| Fine-tuned | 0.956 | 0.989 | 0.966 |

See `docs/writeup.md` for the full discussion, including the important
caveat that this same-split comparison does not establish generalization
to floor types outside the fine-tuning distribution.

## Setup

In [16]:
from pathlib import Path
import gc
import json
import os
import random
import sys

# Prevent the Albumentations update-check warning from adding noise.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm.auto import tqdm

import albumentations as A


# ------------------------------------------------------------------
# Reproducibility and runtime configuration
# ------------------------------------------------------------------

SEED = 42
IMAGE_SIZE = 384

# GTX 1650/Turing-specific stability settings established by diagnostics.
USE_AMP = False
GRAD_CLIP_MAX_NORM = 1.0

# Explicitly clear autocast state as an additional safeguard.
torch.set_autocast_enabled(False)


def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"


# ------------------------------------------------------------------
# Clean-kernel assertions
# ------------------------------------------------------------------

assert USE_AMP is False
assert not torch.is_autocast_enabled(), (
    "Autocast is unexpectedly enabled immediately after reset."
)


print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", __import__("torchvision").__version__)
print("Albumentations:", A.__version__)
print("OpenCV:", cv2.__version__)
print("Device:", DEVICE)
print("USE_AMP:", USE_AMP)
print("Global autocast enabled:", torch.is_autocast_enabled())

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "CUDA capability:",
        torch.cuda.get_device_capability(0),
    )
    print(
        "Allocated GPU memory:",
        round(torch.cuda.memory_allocated() / 1024**2, 2),
        "MiB",
    )

Python: 3.12.3
PyTorch: 2.13.0+cu130
Torchvision: 0.28.0+cu130
Albumentations: 1.4.15
OpenCV: 5.0.0
Device: cuda
USE_AMP: False
Global autocast enabled: False
GPU: NVIDIA GeForce GTX 1650 with Max-Q Design
CUDA capability: (7, 5)
Allocated GPU memory: 52.33 MiB


In [17]:
# ------------------------------------------------------------------
# Project and dataset paths
# ------------------------------------------------------------------

# Repo-relative paths. Run this notebook from the repo root, or adjust
# PROJECT_ROOT to point at wherever this repo is cloned.
#
# Expected layout (see README.md):
#   <repo>/
#     data/floor_seg_data/{images,masks,ignore_masks}/  <- hand-annotated data
#     checkpoints/mobilenetv3_unet_cv5_augv2/fold_0/best.pt  <- source checkpoint
#     artifacts/  <- fine-tuning output (created here)
#
# The hand-annotated data and source checkpoint are not included in this
# repo (large binary files) -- see README.md for how to obtain/place them.
# If you're just reading this notebook rather than running it, these
# paths can be left as-is; they only need to resolve when actually
# executing the notebook.

PROJECT_ROOT = Path(".").resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "floor_seg_data"

IMAGE_DIR = DATA_ROOT / "images"
MASK_DIR = DATA_ROOT / "masks"
IGNORE_MASK_DIR = DATA_ROOT / "ignore_masks"

ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts"
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------------
# Verify directories
# ------------------------------------------------------------------

required_directories = {
    "project root": PROJECT_ROOT,
    "data root": DATA_ROOT,
    "images": IMAGE_DIR,
    "masks": MASK_DIR,
    "ignore masks": IGNORE_MASK_DIR,
}

for name, path in required_directories.items():
    assert path.is_dir(), f"{name} directory not found: {path}"


# ------------------------------------------------------------------
# Basic file-count sanity check
# ------------------------------------------------------------------

image_paths = sorted(IMAGE_DIR.glob("*.jpg"))
mask_paths = sorted(MASK_DIR.glob("*.png"))
ignore_mask_paths = sorted(IGNORE_MASK_DIR.glob("*.png"))

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print()
print("JPEG images:", len(image_paths))
print("PNG masks:", len(mask_paths))
print("PNG ignore masks:", len(ignore_mask_paths))

assert image_paths, f"No .jpg images found in {IMAGE_DIR}"
assert mask_paths, f"No .png masks found in {MASK_DIR}"
assert ignore_mask_paths, (
    f"No .png ignore masks found in {IGNORE_MASK_DIR}"
)

print("\nDataset directories verified.")


Project root: /home/dragoon/peppermint
Data root: /home/dragoon/peppermint/data/floor_seg_data

JPEG images: 164
PNG masks: 164
PNG ignore masks: 164

Dataset directories verified.


In [18]:
# ------------------------------------------------------------------
# Source checkpoint
# ------------------------------------------------------------------

# Points at the highest-IoU fold from the augmented cross-validation run
# (01_cv_augmented.ipynb): fold 0, IoU 0.8020 -- the best of the 5 folds
# (fold 1, IoU 0.8007, was used for an earlier version of this fine-tune;
# fold 0 is the correct one to use going forward, per-fold results are in
# docs/benchmark_report.md).
#
# All 5 CV fold checkpoints are included in this repo under
# checkpoints/fold_0 through checkpoints/fold_4 (each with best.pt,
# last.pt, history.csv, summary.json), plus checkpoints/cv_summary.json
# for the aggregate. The already-fine-tuned checkpoint used for this
# repo's reported results is at checkpoints/Offline_Fine_tuned/best.pt --
# use that directly with scripts/infer_openvino.py or
# scripts/evaluate_openvino_iou.py if you just want to reproduce the
# reported numbers without re-running this notebook.

MOBILENET_CHECKPOINT_PATH = (
    PROJECT_ROOT / "checkpoints" / "fold_0" / "best.pt"
)

assert MOBILENET_CHECKPOINT_PATH.is_file(), (
    f"MobileNet checkpoint not found:\n{MOBILENET_CHECKPOINT_PATH}\n"
    "See the comment above this cell for where to obtain it."
)


# ------------------------------------------------------------------
# Fresh output paths
# ------------------------------------------------------------------

RUN_NAME = "finetune_hand_annotated_mobilenet_clean"

MOBILENET_OUTPUT_DIR = ARTIFACTS_ROOT / RUN_NAME
MOBILENET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MOBILENET_BEST_PATH = MOBILENET_OUTPUT_DIR / "best.pt"
MOBILENET_LAST_PATH = MOBILENET_OUTPUT_DIR / "last.pt"
MOBILENET_HISTORY_PATH = MOBILENET_OUTPUT_DIR / "history.csv"
MOBILENET_SUMMARY_PATH = MOBILENET_OUTPUT_DIR / "summary.json"
MOBILENET_MANIFEST_PATH = MOBILENET_OUTPUT_DIR / "manifest.csv"


# Ensure this is genuinely a new run rather than a checkpoint resume.
old_run_files = [
    MOBILENET_BEST_PATH,
    MOBILENET_LAST_PATH,
    MOBILENET_HISTORY_PATH,
    MOBILENET_SUMMARY_PATH,
    MOBILENET_MANIFEST_PATH,
]

existing_run_files = [path for path in old_run_files if path.exists()]

assert not existing_run_files, (
    "The clean output directory already contains run files:\n"
    + "\n".join(str(path) for path in existing_run_files)
    + "\n\nRename RUN_NAME or remove this directory before continuing."
)


# ------------------------------------------------------------------
# Fine-tuning schedule
# ------------------------------------------------------------------

PHASE1_EPOCHS = 40
PHASE2_EPOCHS = 0

FINETUNE_EPOCHS = PHASE1_EPOCHS + PHASE2_EPOCHS

# Phase 1: last MobileNet encoder stage and decoder.
DECODER_LR = 5e-5
PARTIAL_BACKBONE_LR = 5e-6

# Only used when PHASE2_EPOCHS > 0.
FULL_BACKBONE_LR = 5e-6
PHASE2_DECODER_LR = 1e-5

WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 6

VAL_FRACTION = 0.15
BATCH_SIZE = 4

# Required for this WSL2 environment.
NUM_WORKERS = 0


# ------------------------------------------------------------------
# Configuration checks
# ------------------------------------------------------------------

assert PHASE1_EPOCHS > 0
assert PHASE2_EPOCHS >= 0
assert FINETUNE_EPOCHS > 0
assert NUM_WORKERS == 0
assert USE_AMP is False
assert not torch.is_autocast_enabled()


print("Source checkpoint:", MOBILENET_CHECKPOINT_PATH)
print("Fresh output directory:", MOBILENET_OUTPUT_DIR)
print()
print(
    f"Phase 1: {PHASE1_EPOCHS} epochs "
    f"(decoder LR={DECODER_LR}, backbone LR={PARTIAL_BACKBONE_LR})"
)
print(f"Phase 2: {PHASE2_EPOCHS} epochs")
print("Batch size:", BATCH_SIZE)
print("DataLoader workers:", NUM_WORKERS)
print("AMP enabled:", USE_AMP)
print("Global autocast enabled:", torch.is_autocast_enabled())


Source checkpoint: /mnt/d/best.pt
Fresh output directory: /home/dragoon/peppermint/artifacts/finetune_hand_annotated_mobilenet_clean_5

Phase 1: 40 epochs (decoder LR=5e-05, backbone LR=5e-06)
Phase 2: 0 epochs
Batch size: 4
DataLoader workers: 0
AMP enabled: False
Global autocast enabled: False


In [19]:
# ------------------------------------------------------------------
# Match each image with its floor mask and ignore mask
# ------------------------------------------------------------------

image_files = sorted(IMAGE_DIR.glob("*.jpg"))

assert image_files, f"No JPEG images found in: {IMAGE_DIR}"

rows = []
missing_masks = []
missing_ignore_masks = []

for image_path in image_files:
    sample_id = image_path.stem

    mask_path = MASK_DIR / f"{sample_id}.png"
    ignore_mask_path = IGNORE_MASK_DIR / f"{sample_id}.png"

    if not mask_path.is_file():
        missing_masks.append(sample_id)
        continue

    if not ignore_mask_path.is_file():
        missing_ignore_masks.append(sample_id)
        continue

    rows.append(
        {
            "sample_id": sample_id,
            "image_filename": image_path.name,
            "mask_filename": mask_path.name,
            "ignore_filename": ignore_mask_path.name,
        }
    )


if missing_masks:
    print(f"WARNING: {len(missing_masks)} images lack masks:")
    print(missing_masks[:10])

if missing_ignore_masks:
    print(f"WARNING: {len(missing_ignore_masks)} images lack ignore masks:")
    print(missing_ignore_masks[:10])


manifest = pd.DataFrame(rows)

assert not manifest.empty, (
    "No complete image/mask/ignore-mask triplets were found."
)

assert manifest["sample_id"].is_unique, (
    "Duplicate sample IDs were found in the dataset."
)


# ------------------------------------------------------------------
# Deterministic validation split
# ------------------------------------------------------------------

rng = np.random.default_rng(SEED)
shuffled_indices = rng.permutation(len(manifest))

val_count = max(1, int(len(manifest) * VAL_FRACTION))
val_indices = set(shuffled_indices[:val_count].tolist())

manifest["split"] = [
    "val" if index in val_indices else "train"
    for index in range(len(manifest))
]


# ------------------------------------------------------------------
# Save and report
# ------------------------------------------------------------------

MOBILENET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest.to_csv(MOBILENET_MANIFEST_PATH, index=False)

split_counts = manifest["split"].value_counts()

print("Complete samples:", len(manifest))
print("Training samples:", int(split_counts.get("train", 0)))
print("Validation samples:", int(split_counts.get("val", 0)))
print("Manifest saved to:", MOBILENET_MANIFEST_PATH)

assert split_counts.get("train", 0) > 0
assert split_counts.get("val", 0) > 0

manifest.head()

Complete samples: 164
Training samples: 140
Validation samples: 24
Manifest saved to: /home/dragoon/peppermint/artifacts/finetune_hand_annotated_mobilenet_clean_5/manifest.csv


,sample_id,image_filename,mask_filename,ignore_filename,split
0,hs_hand_hypersim_0000,hs_hand_hypersim_0000.jpg,hs_hand_hypersim_0000.png,hs_hand_hypersim_0000.png,train
1,hs_hand_hypersim_0001,hs_hand_hypersim_0001.jpg,hs_hand_hypersim_0001.png,hs_hand_hypersim_0001.png,train
2,hs_hand_hypersim_0002,hs_hand_hypersim_0002.jpg,hs_hand_hypersim_0002.png,hs_hand_hypersim_0002.png,train
3,hs_hand_hypersim_0003,hs_hand_hypersim_0003.jpg,hs_hand_hypersim_0003.png,hs_hand_hypersim_0003.png,val
4,hs_hand_hypersim_0006,hs_hand_hypersim_0006.jpg,hs_hand_hypersim_0006.png,hs_hand_hypersim_0006.png,train


In [20]:
# ------------------------------------------------------------------
# Image and mask transforms
# ------------------------------------------------------------------

PAD_VALUE = 0
MASK_PAD_VALUE = 0

train_transform = A.Compose(
    [
        A.LongestMaxSize(max_size=IMAGE_SIZE),

        A.PadIfNeeded(
            min_height=IMAGE_SIZE,
            min_width=IMAGE_SIZE,
            border_mode=cv2.BORDER_CONSTANT,
            value=PAD_VALUE,
            mask_value=MASK_PAD_VALUE,
        ),

        # Keep augmentation deliberately light for the small dataset.
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.35),
        A.HueSaturationValue(p=0.20),
        A.GaussianBlur(blur_limit=(3, 5), p=0.10),

        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ]
)

val_transform = A.Compose(
    [
        A.LongestMaxSize(max_size=IMAGE_SIZE),

        A.PadIfNeeded(
            min_height=IMAGE_SIZE,
            min_width=IMAGE_SIZE,
            border_mode=cv2.BORDER_CONSTANT,
            value=PAD_VALUE,
            mask_value=MASK_PAD_VALUE,
        ),

        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ]
)


print("Training transform:", train_transform)
print()
print("Validation transform:", val_transform)

Training transform: Compose([
  LongestMaxSize(p=1.0, max_size=[384], interpolation=1),
  PadIfNeeded(p=1.0, min_height=384, min_width=384, pad_height_divisor=None, pad_width_divisor=None, position=PositionType.CENTER, border_mode=0, value=0.0, mask_value=0.0),
  HorizontalFlip(p=0.5),
  RandomBrightnessContrast(p=0.35, brightness_limit=(-0.2, 0.2), contrast_limit=(-0.2, 0.2), brightness_by_max=True),
  HueSaturationValue(p=0.2, hue_shift_limit=(-20, 20), sat_shift_limit=(-30, 30), val_shift_limit=(-20, 20)),
  GaussianBlur(p=0.1, blur_limit=(3, 5), sigma_limit=(0, 0)),
  Normalize(p=1.0, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0, normalization='standard'),
], p=1.0, bbox_params=None, keypoint_params=None, additional_targets={}, is_check_shapes=True)

Validation transform: Compose([
  LongestMaxSize(p=1.0, max_size=[384], interpolation=1),
  PadIfNeeded(p=1.0, min_height=384, min_width=384, pad_height_divisor=None, pad_width_divisor=None, position=Pos

In [21]:
class FloorDatasetWithIgnore(Dataset):
    """
    Loads:
      - RGB image
      - binary floor mask
      - binary ignore mask

    Ignore-mask pixels are excluded from loss and metric calculations later.
    The same spatial augmentation is applied to both masks and the image.
    """

    def __init__(
        self,
        image_dir,
        mask_dir,
        ignore_dir,
        rows,
        transform,
    ):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.ignore_dir = Path(ignore_dir)
        self.rows = rows.reset_index(drop=True).copy()
        self.transform = transform

        required_columns = {
            "sample_id",
            "image_filename",
            "mask_filename",
            "ignore_filename",
        }

        missing_columns = required_columns - set(self.rows.columns)

        assert not missing_columns, (
            f"Manifest columns missing: {missing_columns}"
        )
        assert len(self.rows) > 0, "Dataset received no rows."

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]

        image_path = self.image_dir / row["image_filename"]
        mask_path = self.mask_dir / row["mask_filename"]
        ignore_path = self.ignore_dir / row["ignore_filename"]

        image_bgr = cv2.imread(
            str(image_path),
            cv2.IMREAD_COLOR,
        )
        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE,
        )
        ignore = cv2.imread(
            str(ignore_path),
            cv2.IMREAD_GRAYSCALE,
        )

        if image_bgr is None:
            raise RuntimeError(
                f"Failed to read image: {image_path}"
            )

        if mask is None:
            raise RuntimeError(
                f"Failed to read floor mask: {mask_path}"
            )

        if ignore is None:
            raise RuntimeError(
                f"Failed to read ignore mask: {ignore_path}"
            )

        image = cv2.cvtColor(
            image_bgr,
            cv2.COLOR_BGR2RGB,
        )

        # Convert masks to binary float arrays.
        mask = (mask > 127).astype(np.float32)
        ignore = (ignore > 127).astype(np.float32)

        augmented = self.transform(
            image=image,
            masks=[mask, ignore],
        )

        transformed_image = augmented["image"]
        transformed_mask, transformed_ignore = augmented["masks"]

        # Explicit contiguous copies prevent negative-stride failures after
        # operations such as HorizontalFlip.
        image_tensor = torch.from_numpy(
            np.ascontiguousarray(
                transformed_image.transpose(2, 0, 1)
            )
        ).float()

        mask_tensor = torch.from_numpy(
            np.ascontiguousarray(transformed_mask)
        ).float().unsqueeze(0)

        ignore_tensor = torch.from_numpy(
            np.ascontiguousarray(transformed_ignore)
        ).float().unsqueeze(0)

        return (
            image_tensor,
            mask_tensor,
            ignore_tensor,
            str(row["sample_id"]),
        )


def make_finetune_loaders(
    manifest,
    image_dir,
    mask_dir,
    ignore_dir,
):
    train_rows = (
        manifest[manifest["split"] == "train"]
        .reset_index(drop=True)
    )

    val_rows = (
        manifest[manifest["split"] == "val"]
        .reset_index(drop=True)
    )

    train_dataset = FloorDatasetWithIgnore(
        image_dir=image_dir,
        mask_dir=mask_dir,
        ignore_dir=ignore_dir,
        rows=train_rows,
        transform=train_transform,
    )

    val_dataset = FloorDatasetWithIgnore(
        image_dir=image_dir,
        mask_dir=mask_dir,
        ignore_dir=ignore_dir,
        rows=val_rows,
        transform=val_transform,
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        persistent_workers=NUM_WORKERS > 0,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
        persistent_workers=NUM_WORKERS > 0,
    )

    return (
        train_loader,
        val_loader,
        train_rows,
        val_rows,
    )


print("FloorDatasetWithIgnore ready.")
print("make_finetune_loaders ready.")
print("NUM_WORKERS:", NUM_WORKERS)

FloorDatasetWithIgnore ready.
make_finetune_loaders ready.
NUM_WORKERS: 0


In [22]:
class ConvBNAct(nn.Sequential):
    def __init__(self, in_channels, out_channels):
        super().__init__(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )


class DecoderBlock(nn.Module):
    def __init__(
        self,
        input_channels,
        skip_channels,
        output_channels,
    ):
        super().__init__()

        self.fusion = ConvBNAct(
            input_channels + skip_channels,
            output_channels,
        )

    def forward(self, decoder_feature, skip_feature):
        decoder_feature = F.interpolate(
            decoder_feature,
            size=skip_feature.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        combined = torch.cat(
            [decoder_feature, skip_feature],
            dim=1,
        )

        return self.fusion(combined)


class MobileNetV3UNet(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()

        # pretrained=False is important here: the source checkpoint supplies
        # all weights, so no torchvision download is required.
        weights = (
            models.MobileNet_V3_Large_Weights.DEFAULT
            if pretrained
            else None
        )

        backbone = models.mobilenet_v3_large(
            weights=weights
        ).features

        # Discover the final feature map at each spatial resolution.
        probe = torch.zeros(
            1,
            3,
            IMAGE_SIZE,
            IMAGE_SIZE,
        )

        stage_information = []
        backbone.eval()

        with torch.no_grad():
            for index, layer in enumerate(backbone):
                probe = layer(probe)

                stage_information.append(
                    (
                        index,
                        int(probe.shape[1]),
                        int(probe.shape[2]),
                        int(probe.shape[3]),
                    )
                )

        last_stage_per_size = {}

        for item in stage_information:
            spatial_size = (item[2], item[3])
            last_stage_per_size[spatial_size] = item

        selected = sorted(
            last_stage_per_size.values(),
            key=lambda item: item[2],
            reverse=True,
        )

        # Exclude the full-resolution input and keep five encoder stages.
        selected = [
            item
            for item in selected
            if item[2] < IMAGE_SIZE
        ][:5]

        assert len(selected) == 5, (
            "Unexpected MobileNet feature structure:\n"
            f"{stage_information}"
        )

        self.stage_ids = [
            item[0]
            for item in selected
        ]

        channels = [
            item[1]
            for item in selected
        ]

        self.backbone = backbone

        # Possible values: "frozen", "partial", or "full".
        self.backbone_mode = "full"

        c1, c2, c3, c4, c5 = channels

        self.center = ConvBNAct(c5, 256)
        self.decoder4 = DecoderBlock(256, c4, 160)
        self.decoder3 = DecoderBlock(160, c3, 96)
        self.decoder2 = DecoderBlock(96, c2, 64)
        self.decoder1 = DecoderBlock(64, c1, 32)
        self.head = nn.Conv2d(
            32,
            1,
            kernel_size=1,
        )

        print(
            "Selected MobileNet stages:",
            list(zip(self.stage_ids, channels)),
        )

    def freeze_all_backbone(self):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

        self.backbone_mode = "frozen"
        self.backbone.eval()

    def partial_unfreeze_backbone(self):
        """
        Unfreeze only the deepest MobileNet encoder stage.

        The complete backbone stays in evaluation mode so that BatchNorm
        running statistics do not drift on the small fine-tuning dataset.
        Trainable convolution parameters still receive gradients in eval mode.
        """
        last_stage_start = self.stage_ids[-2]

        for index, layer in enumerate(self.backbone):
            requires_grad = index >= last_stage_start

            for parameter in layer.parameters():
                parameter.requires_grad = requires_grad

        self.backbone_mode = "partial"
        self.backbone.eval()

    def full_unfreeze_backbone(self):
        for parameter in self.backbone.parameters():
            parameter.requires_grad = True

        self.backbone_mode = "full"

    def train(self, mode=True):
        super().train(mode)

        # Preserve frozen encoder BatchNorm statistics.
        if mode and self.backbone_mode in ("frozen", "partial"):
            self.backbone.eval()

        return self

    def decoder_parameters(self):
        decoder_modules = [
            self.center,
            self.decoder4,
            self.decoder3,
            self.decoder2,
            self.decoder1,
            self.head,
        ]

        for module in decoder_modules:
            yield from module.parameters()

    def last_stage_backbone_parameters(self):
        last_stage_start = self.stage_ids[-2]

        for index, layer in enumerate(self.backbone):
            if index >= last_stage_start:
                yield from layer.parameters()

    def forward(self, image):
        input_size = image.shape[-2:]

        features = []
        feature = image
        wanted_stage_ids = set(self.stage_ids)

        for index, layer in enumerate(self.backbone):
            feature = layer(feature)

            if index in wanted_stage_ids:
                features.append(feature)

        assert len(features) == 5, (
            f"Expected 5 encoder features, got {len(features)}"
        )

        feature1, feature2, feature3, feature4, feature5 = features

        decoded = self.center(feature5)
        decoded = self.decoder4(decoded, feature4)
        decoded = self.decoder3(decoded, feature3)
        decoded = self.decoder2(decoded, feature2)
        decoded = self.decoder1(decoded, feature1)

        logits = self.head(decoded)

        return F.interpolate(
            logits,
            size=input_size,
            mode="bilinear",
            align_corners=False,
        )


print("MobileNetV3UNet defined.")

MobileNetV3UNet defined.


### Load source checkpoint

`extract_model_state_dict` handles a few common checkpoint-saving conventions (raw state dict, `{'model': ...}`, DataParallel's `module.` prefix) so this works regardless of exactly how the upstream training notebook saved it.

In [23]:
def extract_model_state_dict(checkpoint):
    if not isinstance(checkpoint, dict):
        raise TypeError(
            f"Unsupported checkpoint type: {type(checkpoint)}"
        )

    for key in ("model", "model_state_dict", "state_dict"):
        value = checkpoint.get(key)

        if isinstance(value, dict):
            state_dict = value
            break
    else:
        # The checkpoint itself may already be a state_dict.
        state_dict = checkpoint

    # Handle checkpoints saved through DataParallel.
    if state_dict and all(
        key.startswith("module.")
        for key in state_dict
    ):
        state_dict = {
            key.removeprefix("module."): value
            for key, value in state_dict.items()
        }

    return state_dict


seed_everything(SEED)

assert USE_AMP is False
assert not torch.is_autocast_enabled()

mobilenet_checkpoint = torch.load(
    MOBILENET_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

mobilenet_state_dict = extract_model_state_dict(
    mobilenet_checkpoint
)

mobilenet_model = MobileNetV3UNet(
    pretrained=False
)

# strict=True confirms that the architecture exactly matches the checkpoint.
mobilenet_model.load_state_dict(
    mobilenet_state_dict,
    strict=True,
)

mobilenet_model = mobilenet_model.to(DEVICE)
mobilenet_model.eval()

(
    mobilenet_train_loader,
    mobilenet_val_loader,
    mobilenet_train_rows,
    mobilenet_val_rows,
) = make_finetune_loaders(
    manifest=manifest,
    image_dir=IMAGE_DIR,
    mask_dir=MASK_DIR,
    ignore_dir=IGNORE_MASK_DIR,
)

print("Loaded checkpoint:", MOBILENET_CHECKPOINT_PATH)

if isinstance(mobilenet_checkpoint, dict):
    for key in ("fold", "epoch", "metrics"):
        if key in mobilenet_checkpoint:
            print(f"Source {key}:", mobilenet_checkpoint[key])

print()
print("Train samples:", len(mobilenet_train_rows))
print("Validation samples:", len(mobilenet_val_rows))
print("Train batches:", len(mobilenet_train_loader))
print("Validation batches:", len(mobilenet_val_loader))
print("Model device:", next(mobilenet_model.parameters()).device)
print("Model dtype:", next(mobilenet_model.parameters()).dtype)
print("Global autocast enabled:", torch.is_autocast_enabled())

del mobilenet_checkpoint
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

Selected MobileNet stages: [(1, 16), (3, 24), (6, 40), (12, 112), (16, 960)]
Loaded checkpoint: /mnt/d/best.pt
Source fold: 0
Source epoch: 13
Source metrics: {'val_loss': 0.12593859418673548, 'iou': 0.8020104385964495, 'dice': 0.8901285158160567, 'precision': 0.8847309889622276, 'recall': 0.895592304895522}

Train samples: 140
Validation samples: 24
Train batches: 35
Validation batches: 6
Model device: cuda:0
Model dtype: torch.float32
Global autocast enabled: False


## Loss and training loop

In [24]:
FP_WEIGHT = 0.80
FN_WEIGHT = 0.20

BCE_WEIGHT = 0.50
TVERSKY_WEIGHT = 0.50
TVERSKY_SMOOTH = 1.0

# A clean run should not silently accept broken validation batches.
STRICT_FINITE_VALIDATION = True

assert 0.0 <= FP_WEIGHT <= 1.0
assert 0.0 <= FN_WEIGHT <= 1.0
assert np.isclose(FP_WEIGHT + FN_WEIGHT, 1.0)
assert np.isclose(BCE_WEIGHT + TVERSKY_WEIGHT, 1.0)


class IgnoreAwareTverskyBCELoss(nn.Module):
    def __init__(
        self,
        fp_weight=0.80,
        fn_weight=0.20,
        bce_weight=0.50,
        tversky_weight=0.50,
        smooth=1.0,
    ):
        super().__init__()

        self.fp_weight = float(fp_weight)
        self.fn_weight = float(fn_weight)
        self.bce_weight = float(bce_weight)
        self.tversky_weight = float(tversky_weight)
        self.smooth = float(smooth)

    def forward(
        self,
        logits,
        targets,
        ignore,
    ):
        if (
            logits.shape != targets.shape
            or logits.shape != ignore.shape
        ):
            raise ValueError(
                "logits, targets, and ignore masks must have "
                "identical shapes; received "
                f"{logits.shape}, {targets.shape}, {ignore.shape}"
            )

        valid = (
            ignore < 0.5
        ).to(dtype=logits.dtype)

        valid_count = valid.sum()

        if valid_count.item() == 0:
            raise RuntimeError(
                "Loss batch contains no valid pixels."
            )

        # Standard BCE, excluding uncertain pixels.
        bce_map = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )

        bce_loss = (
            bce_map * valid
        ).sum() / valid_count

        probabilities = torch.sigmoid(logits)
        reduce_dimensions = (1, 2, 3)

        true_positive = (
            probabilities
            * targets
            * valid
        ).sum(dim=reduce_dimensions)

        false_positive = (
            probabilities
            * (1.0 - targets)
            * valid
        ).sum(dim=reduce_dimensions)

        false_negative = (
            (1.0 - probabilities)
            * targets
            * valid
        ).sum(dim=reduce_dimensions)

        tversky_score = (
            true_positive + self.smooth
        ) / (
            true_positive
            + self.fp_weight * false_positive
            + self.fn_weight * false_negative
            + self.smooth
        )

        tversky_loss = (
            1.0 - tversky_score.mean()
        )

        return (
            self.bce_weight * bce_loss
            + self.tversky_weight * tversky_loss
        )


finetune_criterion = IgnoreAwareTverskyBCELoss(
    fp_weight=FP_WEIGHT,
    fn_weight=FN_WEIGHT,
    bce_weight=BCE_WEIGHT,
    tversky_weight=TVERSKY_WEIGHT,
    smooth=TVERSKY_SMOOTH,
)

print("Loss ready.")
print("FP_WEIGHT:", FP_WEIGHT)
print("FN_WEIGHT:", FN_WEIGHT)
print("USE_AMP:", USE_AMP)
print("GRAD_CLIP_MAX_NORM:", GRAD_CLIP_MAX_NORM)
print("Strict finite validation:", STRICT_FINITE_VALIDATION)

Loss ready.
FP_WEIGHT: 0.8
FN_WEIGHT: 0.2
USE_AMP: False
GRAD_CLIP_MAX_NORM: 1.0
Strict finite validation: True


In [25]:
from contextlib import nullcontext


def save_json(value, path):
    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            value,
            file,
            indent=2,
        )


def make_autocast_context():
    if USE_AMP and DEVICE.type == "cuda":
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=True,
        )

    # Do not enter torch.autocast at all when AMP is disabled.
    return nullcontext()


def make_grad_scaler():
    enabled = bool(
        USE_AMP
        and DEVICE.type == "cuda"
    )

    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=enabled,
        )
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(
            enabled=enabled,
        )


def train_one_finetune_epoch(
    model,
    loader,
    optimizer,
    scaler,
    description="Training",
):
    if (
        not USE_AMP
        and torch.is_autocast_enabled()
    ):
        raise RuntimeError(
            "Global autocast became enabled despite USE_AMP=False. "
            "Restart the kernel before continuing."
        )

    model.train()

    losses = []
    skipped_batches = 0
    skipped_sample_ids = []

    trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    progress = tqdm(
        loader,
        desc=description,
        leave=False,
    )

    for batch in progress:
        images, targets, ignore, sample_ids = batch

        images = images.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )
        targets = targets.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )
        ignore = ignore.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        inputs_are_finite = (
            torch.isfinite(images).all()
            and torch.isfinite(targets).all()
            and torch.isfinite(ignore).all()
        )

        if not inputs_are_finite:
            skipped_batches += 1
            skipped_sample_ids.extend(
                list(sample_ids)
            )
            continue

        optimizer.zero_grad(
            set_to_none=True
        )

        with make_autocast_context():
            logits = model(images)

            if torch.isfinite(logits).all():
                loss = finetune_criterion(
                    logits,
                    targets,
                    ignore,
                )
            else:
                loss = None

        if (
            loss is None
            or not torch.isfinite(loss)
        ):
            optimizer.zero_grad(
                set_to_none=True
            )

            skipped_batches += 1
            skipped_sample_ids.extend(
                list(sample_ids)
            )
            continue

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        gradient_norm = (
            torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=GRAD_CLIP_MAX_NORM,
            )
        )

        if not torch.isfinite(gradient_norm):
            optimizer.zero_grad(
                set_to_none=True
            )

            scaler.update()

            skipped_batches += 1
            skipped_sample_ids.extend(
                list(sample_ids)
            )
            continue

        scaler.step(optimizer)
        scaler.update()

        loss_value = float(
            loss.detach().item()
        )
        losses.append(loss_value)

        progress.set_postfix(
            loss=f"{loss_value:.4f}",
            grad=f"{float(gradient_norm):.3f}",
            skipped=skipped_batches,
        )

    if skipped_batches:
        print(
            f"[{description}] skipped "
            f"{skipped_batches} non-finite batch(es)."
        )
        print(
            "Affected sample_ids:",
            skipped_sample_ids,
        )

    if not losses:
        raise RuntimeError(
            f"{description} produced no finite optimization steps."
        )

    return float(np.mean(losses))


@torch.no_grad()
def evaluate_finetune(
    model,
    loader,
    threshold=0.5,
    strict_nonfinite=STRICT_FINITE_VALIDATION,
):
    if (
        not USE_AMP
        and torch.is_autocast_enabled()
    ):
        raise RuntimeError(
            "Global autocast became enabled despite USE_AMP=False. "
            "Restart the kernel before continuing."
        )

    model.eval()

    losses = []

    intersection = 0
    union = 0

    true_positive = 0
    false_positive = 0
    false_negative = 0

    valid_pixels = 0

    skipped_batches = 0
    skipped_sample_ids = []

    progress = tqdm(
        loader,
        desc="Validation",
        leave=False,
    )

    for batch in progress:
        images, targets, ignore, sample_ids = batch

        images = images.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )
        targets = targets.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )
        ignore = ignore.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        inputs_are_finite = (
            torch.isfinite(images).all()
            and torch.isfinite(targets).all()
            and torch.isfinite(ignore).all()
        )

        if not inputs_are_finite:
            skipped_batches += 1
            skipped_sample_ids.extend(
                list(sample_ids)
            )
            continue

        with make_autocast_context():
            logits = model(images)

            if torch.isfinite(logits).all():
                loss = finetune_criterion(
                    logits,
                    targets,
                    ignore,
                )
            else:
                loss = None

        if (
            loss is None
            or not torch.isfinite(loss)
        ):
            skipped_batches += 1
            skipped_sample_ids.extend(
                list(sample_ids)
            )
            continue

        losses.append(
            float(loss.item())
        )

        valid = ignore < 0.5

        predictions = (
            torch.sigmoid(logits) >= threshold
        ) & valid

        ground_truth = (
            targets >= 0.5
        ) & valid

        batch_true_positive = (
            predictions & ground_truth
        )

        batch_false_positive = (
            predictions
            & (~ground_truth)
            & valid
        )

        batch_false_negative = (
            (~predictions)
            & ground_truth
            & valid
        )

        intersection += int(
            batch_true_positive.sum().item()
        )

        union += int(
            (
                predictions | ground_truth
            ).sum().item()
        )

        true_positive += int(
            batch_true_positive.sum().item()
        )

        false_positive += int(
            batch_false_positive.sum().item()
        )

        false_negative += int(
            batch_false_negative.sum().item()
        )

        valid_pixels += int(
            valid.sum().item()
        )

    if skipped_batches:
        message = (
            f"Validation skipped {skipped_batches} "
            "non-finite batch(es); "
            f"sample_ids={skipped_sample_ids}"
        )

        if strict_nonfinite:
            raise RuntimeError(message)

        print(message)

    if not losses:
        raise RuntimeError(
            "Validation produced no finite batches."
        )

    metrics = {
        "val_loss": float(np.mean(losses)),

        "iou": float(
            intersection
            / max(union, 1)
        ),

        "dice": float(
            2 * true_positive
            / max(
                2 * true_positive
                + false_positive
                + false_negative,
                1,
            )
        ),

        "precision": float(
            true_positive
            / max(
                true_positive
                + false_positive,
                1,
            )
        ),

        "recall": float(
            true_positive
            / max(
                true_positive
                + false_negative,
                1,
            )
        ),

        "true_positive_pixels": int(
            true_positive
        ),

        "false_positive_pixels": int(
            false_positive
        ),

        "false_negative_pixels": int(
            false_negative
        ),

        "valid_pixels": int(
            valid_pixels
        ),

        "skipped_batches": int(
            skipped_batches
        ),
    }

    finite_metric_names = (
        "val_loss",
        "iou",
        "dice",
        "precision",
        "recall",
    )

    metrics_are_finite = all(
        np.isfinite(metrics[name])
        for name in finite_metric_names
    )

    if not metrics_are_finite:
        raise RuntimeError(
            f"Validation returned non-finite metrics: {metrics}"
        )

    return metrics


print("Training and evaluation helpers ready.")

Training and evaluation helpers ready.


In [26]:
def build_phase1_optimizer(model):
    """
    Train only:
      - deepest MobileNet encoder stage
      - complete UNet decoder
    """
    model.freeze_all_backbone()
    model.partial_unfreeze_backbone()

    backbone_parameters = list(
        model.last_stage_backbone_parameters()
    )

    decoder_parameters = list(
        model.decoder_parameters()
    )

    assert backbone_parameters
    assert decoder_parameters

    return torch.optim.AdamW(
        [
            {
                "params": backbone_parameters,
                "lr": PARTIAL_BACKBONE_LR,
                "name": "last_backbone_stage",
            },
            {
                "params": decoder_parameters,
                "lr": DECODER_LR,
                "name": "decoder",
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )


def build_phase2_optimizer(model):
    model.full_unfreeze_backbone()

    return torch.optim.AdamW(
        [
            {
                "params": list(
                    model.backbone.parameters()
                ),
                "lr": FULL_BACKBONE_LR,
                "name": "full_backbone",
            },
            {
                "params": list(
                    model.decoder_parameters()
                ),
                "lr": PHASE2_DECODER_LR,
                "name": "decoder",
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )


def optimizer_learning_rates(optimizer):
    return {
        group.get(
            "name",
            f"group_{index}",
        ): float(group["lr"])
        for index, group
        in enumerate(optimizer.param_groups)
    }


def run_finetune(
    model,
    train_loader,
    val_loader,
    train_rows,
    val_rows,
    best_path,
    last_path,
    history_path,
    summary_path,
    source_checkpoint_path,
    source_metrics,
    model_name="MobileNet",
):
    best_path = Path(best_path)
    last_path = Path(last_path)
    history_path = Path(history_path)
    summary_path = Path(summary_path)

    # Defensive directory recreation.
    for path in (
        best_path,
        last_path,
        history_path,
        summary_path,
    ):
        path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

    total_epochs = (
        PHASE1_EPOCHS
        + PHASE2_EPOCHS
    )

    assert total_epochs > 0

    scaler = make_grad_scaler()

    history = []
    best_iou = -1.0
    best_epoch = -1
    best_metrics = None
    patience_counter = 0

    start_epoch = 1
    phase = 1

    # This permits recovery if training is interrupted after this point.
    if last_path.exists():
        print(
            f"Resuming interrupted {model_name} run from:",
            last_path,
        )

        resume_state = torch.load(
            last_path,
            map_location=DEVICE,
            weights_only=False,
        )

        model.load_state_dict(
            resume_state["model"],
            strict=True,
        )

        start_epoch = (
            int(resume_state["epoch"])
            + 1
        )

        saved_phase = int(
            resume_state["phase"]
        )

        best_iou = float(
            resume_state["best_iou"]
        )

        best_epoch = int(
            resume_state["best_epoch"]
        )

        best_metrics = resume_state.get(
            "best_metrics"
        )

        patience_counter = int(
            resume_state.get(
                "patience_counter",
                0,
            )
        )

        history = list(
            resume_state.get(
                "history",
                [],
            )
        )

        if start_epoch > total_epochs:
            print(
                "The saved run has already completed "
                "all configured epochs."
            )

            best_checkpoint = torch.load(
                best_path,
                map_location=DEVICE,
                weights_only=False,
            )

            model.load_state_dict(
                best_checkpoint["model"],
                strict=True,
            )

            if summary_path.exists():
                with summary_path.open(
                    "r",
                    encoding="utf-8",
                ) as file:
                    summary = json.load(file)
            else:
                summary = {
                    "model_name": model_name,
                    "best_epoch": best_epoch,
                    "best_iou": best_metrics["iou"],
                    "best_precision": best_metrics[
                        "precision"
                    ],
                    "best_recall": best_metrics[
                        "recall"
                    ],
                }

            return summary, best_metrics

        phase = (
            1
            if start_epoch <= PHASE1_EPOCHS
            else 2
        )

        optimizer = (
            build_phase1_optimizer(model)
            if phase == 1
            else build_phase2_optimizer(model)
        )

        phase_epoch_total = (
            PHASE1_EPOCHS
            if phase == 1
            else PHASE2_EPOCHS
        )

        scheduler = (
            torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=max(
                    phase_epoch_total,
                    1,
                ),
            )
        )

        if phase == saved_phase:
            optimizer.load_state_dict(
                resume_state["optimizer"]
            )

            scheduler.load_state_dict(
                resume_state["scheduler"]
            )

            scaler_state = resume_state.get(
                "scaler"
            )

            if scaler_state is not None:
                scaler.load_state_dict(
                    scaler_state
                )
        else:
            patience_counter = 0

        print(
            f"Continuing at epoch {start_epoch}, "
            f"phase {phase}."
        )

    else:
        optimizer = build_phase1_optimizer(
            model
        )

        scheduler = (
            torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max=max(
                    PHASE1_EPOCHS,
                    1,
                ),
            )
        )

        print(
            f"{model_name} phase 1: "
            f"{PHASE1_EPOCHS} epochs"
        )
        print(
            "Last encoder stage LR:",
            PARTIAL_BACKBONE_LR,
        )
        print(
            "Decoder LR:",
            DECODER_LR,
        )

    for epoch in range(
        start_epoch,
        total_epochs + 1,
    ):
        if (
            PHASE2_EPOCHS > 0
            and epoch == PHASE1_EPOCHS + 1
            and phase == 1
        ):
            phase = 2
            patience_counter = 0

            optimizer = (
                build_phase2_optimizer(model)
            )

            scheduler = (
                torch.optim.lr_scheduler.CosineAnnealingLR(
                    optimizer,
                    T_max=max(
                        PHASE2_EPOCHS,
                        1,
                    ),
                )
            )

            print(
                f"{model_name} phase 2: "
                f"{PHASE2_EPOCHS} epochs"
            )

        phase_epoch = (
            epoch
            if phase == 1
            else epoch - PHASE1_EPOCHS
        )

        phase_total = (
            PHASE1_EPOCHS
            if phase == 1
            else PHASE2_EPOCHS
        )

        learning_rates = (
            optimizer_learning_rates(
                optimizer
            )
        )

        train_loss = (
            train_one_finetune_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                scaler=scaler,
                description=(
                    f"{model_name} phase {phase} "
                    f"epoch {phase_epoch}/{phase_total}"
                ),
            )
        )

        metrics = evaluate_finetune(
            model,
            val_loader,
            threshold=0.5,
        )

        row = {
            "epoch": int(epoch),
            "phase": int(phase),
            "train_loss": float(train_loss),

            **{
                f"lr_{name}": value
                for name, value
                in learning_rates.items()
            },

            **metrics,
        }

        history.append(row)

        print(
            json.dumps(
                row,
                indent=2,
            )
        )

        if metrics["iou"] > best_iou:
            best_iou = float(
                metrics["iou"]
            )

            best_epoch = int(epoch)
            best_metrics = dict(metrics)
            patience_counter = 0

            torch.save(
                {
                    "model": model.state_dict(),
                    "epoch": int(epoch),
                    "phase": int(phase),
                    "metrics": best_metrics,
                    "source_checkpoint": str(
                        source_checkpoint_path
                    ),
                    "fp_weight": FP_WEIGHT,
                    "fn_weight": FN_WEIGHT,
                },
                best_path,
            )

        else:
            patience_counter += 1

        scheduler.step()

        pd.DataFrame(history).to_csv(
            history_path,
            index=False,
        )

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "epoch": int(epoch),
                "phase": int(phase),
                "best_iou": float(best_iou),
                "best_epoch": int(best_epoch),
                "best_metrics": best_metrics,
                "patience_counter": int(
                    patience_counter
                ),
                "history": history,
            },
            last_path,
        )

        if (
            patience_counter
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                f"Early stopping {model_name} "
                f"at epoch {epoch}: no IoU "
                f"improvement for "
                f"{patience_counter} epochs."
            )
            break

    if best_metrics is None:
        raise RuntimeError(
            f"No finite checkpoint was produced "
            f"for {model_name}."
        )

    # Reload and independently confirm the best checkpoint.
    best_checkpoint = torch.load(
        best_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        best_checkpoint["model"],
        strict=True,
    )
    model.eval()

    confirmed_best_metrics = evaluate_finetune(
        model,
        val_loader,
        threshold=0.5,
    )

    summary = {
        "model_name": model_name,

        "source_checkpoint": str(
            source_checkpoint_path
        ),

        "phase1_epochs": int(
            PHASE1_EPOCHS
        ),

        "phase2_epochs": int(
            PHASE2_EPOCHS
        ),

        "fp_weight": float(
            FP_WEIGHT
        ),

        "fn_weight": float(
            FN_WEIGHT
        ),

        "train_samples": int(
            len(train_rows)
        ),

        "val_samples": int(
            len(val_rows)
        ),

        "best_epoch": int(
            best_epoch
        ),

        "best_iou": float(
            confirmed_best_metrics["iou"]
        ),

        "best_dice": float(
            confirmed_best_metrics["dice"]
        ),

        "best_precision": float(
            confirmed_best_metrics["precision"]
        ),

        "best_recall": float(
            confirmed_best_metrics["recall"]
        ),

        "best_val_loss": float(
            confirmed_best_metrics["val_loss"]
        ),

        "source_same_split_iou": float(
            source_metrics["iou"]
        ),

        "source_same_split_precision": float(
            source_metrics["precision"]
        ),

        "source_same_split_recall": float(
            source_metrics["recall"]
        ),

        "iou_delta_same_split": float(
            confirmed_best_metrics["iou"]
            - source_metrics["iou"]
        ),

        "precision_delta_same_split": float(
            confirmed_best_metrics["precision"]
            - source_metrics["precision"]
        ),

        "recall_delta_same_split": float(
            confirmed_best_metrics["recall"]
            - source_metrics["recall"]
        ),

        "decoder_lr": float(
            DECODER_LR
        ),

        "partial_backbone_lr": float(
            PARTIAL_BACKBONE_LR
        ),

        "full_backbone_lr": float(
            FULL_BACKBONE_LR
        ),

        "phase2_decoder_lr": float(
            PHASE2_DECODER_LR
        ),

        "max_epochs": int(
            total_epochs
        ),
    }

    save_json(
        summary,
        summary_path,
    )

    print(
        f"{model_name} best epoch:",
        best_epoch,
    )

    print(
        json.dumps(
            summary,
            indent=2,
        )
    )

    print(
        "Best checkpoint:",
        best_path,
    )

    return (
        summary,
        confirmed_best_metrics,
    )


print(
    "Optimizer builders and "
    "run_finetune are ready."
)

Optimizer builders and run_finetune are ready.


## Evaluate the source checkpoint on this split

Establishes the "before" numbers for the comparison table, on the exact same held-out split the fine-tuned model will later be evaluated on.

In [27]:
assert not torch.is_autocast_enabled()

mobilenet_model.eval()

mobilenet_source_metrics = evaluate_finetune(
    mobilenet_model,
    mobilenet_val_loader,
    threshold=0.5,
)

print(
    "Source checkpoint metrics on the current "
    "validation split:"
)

print(
    json.dumps(
        mobilenet_source_metrics,
        indent=2,
    )
)

for metric_name in (
    "val_loss",
    "iou",
    "dice",
    "precision",
    "recall",
):
    metric_value = mobilenet_source_metrics[
        metric_name
    ]

    assert np.isfinite(metric_value), (
        f"Source metric {metric_name} is non-finite: "
        f"{metric_value}"
    )

assert (
    mobilenet_source_metrics["skipped_batches"]
    == 0
)

print(
    "Source checkpoint forward pass and "
    "validation are fully finite."
)

Validation:   0%|          | 0/6 [00:00<?, ?it/s]

Source checkpoint metrics on the current validation split:
{
  "val_loss": 0.08372095661858718,
  "iou": 0.8844536791831272,
  "dice": 0.938684446270412,
  "precision": 0.9532310939867957,
  "recall": 0.9245750992908942,
  "true_positive_pixels": 690003,
  "false_positive_pixels": 33854,
  "false_negative_pixels": 56289,
  "valid_pixels": 3538944,
  "skipped_batches": 0
}
Source checkpoint forward pass and validation are fully finite.


## Fine-tune

In [28]:
assert USE_AMP is False

assert not torch.is_autocast_enabled(), (
    "Autocast is globally enabled. Restart the kernel "
    "and rerun from Cell 1."
)

mobilenet_summary, mobilenet_best_metrics = run_finetune(
    model=mobilenet_model,

    train_loader=mobilenet_train_loader,
    val_loader=mobilenet_val_loader,

    train_rows=mobilenet_train_rows,
    val_rows=mobilenet_val_rows,

    best_path=MOBILENET_BEST_PATH,
    last_path=MOBILENET_LAST_PATH,
    history_path=MOBILENET_HISTORY_PATH,
    summary_path=MOBILENET_SUMMARY_PATH,

    source_checkpoint_path=(
        MOBILENET_CHECKPOINT_PATH
    ),

    source_metrics=(
        mobilenet_source_metrics
    ),

    model_name="MobileNetV3-UNet",
)

MobileNetV3-UNet phase 1: 40 epochs
Last encoder stage LR: 5e-06
Decoder LR: 5e-05


MobileNetV3-UNet phase 1 epoch 1/40:   0%|          | 0/35 [00:00<?, ?it/s]

Validation:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "epoch": 1,
  "phase": 1,
  "train_loss": 0.09003978515309947,
  "lr_last_backbone_stage": 5e-06,
  "lr_decoder": 5e-05,
  "val_loss": 0.051322865299880505,
  "iou": 0.92895421391551,
  "dice": 0.9631687545655753,
  "precision": 0.9799157529270173,
  "recall": 0.9469845583230156,
  "true_positive_pixels": 706727,
  "false_positive_pixels": 14485,
  "false_negative_pixels": 39565,
  "valid_pixels": 3538944,
  "skipped_batches": 0
}


MobileNetV3-UNet phase 1 epoch 2/40:   0%|          | 0/35 [00:00<?, ?it/s]

...

MobileNetV3-UNet phase 1 epoch 40/40:   0%|          | 0/35 [00:00<?, ?it/s]

Validation:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "epoch": 40,
  "phase": 1,
  "train_loss": 0.022605075793606894,
  "lr_last_backbone_stage": 7.706665667180087e-09,
  "lr_decoder": 7.706665667180087e-08,
  "val_loss": 0.027337700438996155,
  "iou": 0.9570586923391443,
  "dice": 0.9780582422852476,
  "precision": 0.9903946676118138,
  "recall": 0.966025362726654,
  "true_positive_pixels": 720937,
  "false_positive_pixels": 6992,
  "false_negative_pixels": 25355,
  "valid_pixels": 3538944,
  "skipped_batches": 0
}


Validation:   0%|          | 0/6 [00:00<?, ?it/s]

MobileNetV3-UNet best epoch: 37
{
  "model_name": "MobileNetV3-UNet",
  "source_checkpoint": "/mnt/d/best.pt",
  "phase1_epochs": 40,
  "phase2_epochs": 0,
  "fp_weight": 0.8,
  "fn_weight": 0.2,
  "train_samples": 140,
  "val_samples": 24,
  "best_epoch": 37,
  "best_iou": 0.9588201453344256,
  "best_dice": 0.9789772150528175,
  "best_precision": 0.988917720982503,
  "best_recall": 0.96923456234289,
  "best_val_loss": 0.02609055256471038,
  "source_same_split_iou": 0.8844536791831272,
  "source_same_split_precision": 0.9532310939867957,
  "source_same_split_recall": 0.9245750992908942,
  "iou_delta_same_split": 0.07436646615129838,
  "precision_delta_same_split": 0.03568662699570724,
  "recall_delta_same_split": 0.04465946305199575,
  "decoder_lr": 5e-05,
  "partial_backbone_lr": 5e-06,
  "full_backbone_lr": 5e-06,
  "phase2_decoder_lr": 1e-05,
  "max_epochs": 40
}
Best checkpoint: /home/dragoon/peppermint/artifacts/finetune_hand_annotated_mobilenet_clean_5/best.pt


## Results

Same-split before/after comparison, and a decision-threshold sweep (useful for trading precision against recall post-hoc, without retraining, if the deployed threshold needs adjusting).

In [29]:
history_df = pd.read_csv(
    MOBILENET_HISTORY_PATH
)

finite_columns = [
    "train_loss",
    "val_loss",
    "iou",
    "dice",
    "precision",
    "recall",
]

assert not history_df.empty

assert np.isfinite(
    history_df[
        finite_columns
    ].to_numpy()
).all(), (
    "The saved history contains NaN or infinity."
)

assert (
    history_df["skipped_batches"] == 0
).all(), (
    "At least one validation batch was skipped."
)

reported_source_reference = {
    "iou": 0.80,
    "precision": 0.89,
    "recall": 0.89,
}

comparison_df = pd.DataFrame(
    [
        {
            "evaluation": (
                "Source checkpoint on current val split"
            ),
            "iou": mobilenet_source_metrics[
                "iou"
            ],
            "precision": mobilenet_source_metrics[
                "precision"
            ],
            "recall": mobilenet_source_metrics[
                "recall"
            ],
        },
        {
            "evaluation": (
                "Fine-tuned best on current val split"
            ),
            "iou": mobilenet_best_metrics[
                "iou"
            ],
            "precision": mobilenet_best_metrics[
                "precision"
            ],
            "recall": mobilenet_best_metrics[
                "recall"
            ],
        },
        {
            "evaluation": (
                "Previously reported source reference"
            ),
            **reported_source_reference,
        },
    ]
)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.4f}"
        ),
    )
)

same_split_iou_delta = (
    mobilenet_best_metrics["iou"]
    - mobilenet_source_metrics["iou"]
)

same_split_precision_delta = (
    mobilenet_best_metrics["precision"]
    - mobilenet_source_metrics["precision"]
)

same_split_recall_delta = (
    mobilenet_best_metrics["recall"]
    - mobilenet_source_metrics["recall"]
)

reported_precision_delta = (
    mobilenet_best_metrics["precision"]
    - reported_source_reference["precision"]
)

print(
    "\nSame-split deltas, "
    "fine-tuned minus source:"
)

print(
    f"  IoU:       "
    f"{same_split_iou_delta:+.4f}"
)

print(
    f"  Precision: "
    f"{same_split_precision_delta:+.4f}"
)

print(
    f"  Recall:    "
    f"{same_split_recall_delta:+.4f}"
)

print(
    "  Precision versus the previously "
    f"reported 0.89: {reported_precision_delta:+.4f}"
)

if same_split_precision_delta > 0:
    print(
        "\nResult: precision improved on the same "
        "held-out images. This is consistent with "
        "the FP-biased loss reducing false-positive "
        "floor pixels."
    )

elif same_split_precision_delta < 0:
    print(
        "\nResult: precision did not improve on the "
        "same held-out images. The FP-biased "
        "fine-tune did not achieve its primary objective."
    )

else:
    print(
        "\nResult: precision was unchanged on the "
        "same held-out images."
    )

print(
    "\nThis comparison measures the complete "
    "fine-tuning procedure. It does not isolate "
    "FP_WEIGHT causally; a neutral-weight ablation "
    "would be needed to prove that the weighting "
    "itself caused the difference."
)

best_history_row = history_df.loc[
    history_df["iou"].idxmax(),
    [
        "epoch",
        "phase",
        "train_loss",
        "val_loss",
        "iou",
        "precision",
        "recall",
    ],
]

print("\nBest epoch row:")
print(best_history_row.to_string())

                            evaluation    iou  precision  recall
Source checkpoint on current val split 0.8845     0.9532  0.9246
  Fine-tuned best on current val split 0.9588     0.9889  0.9692
  Previously reported source reference 0.8000     0.8900  0.8900

Same-split deltas, fine-tuned minus source:
  IoU:       +0.0744
  Precision: +0.0357
  Recall:    +0.0447
  Precision versus the previously reported 0.89: +0.0989

Result: precision improved on the same held-out images. This is consistent with the FP-biased loss reducing false-positive floor pixels.

This comparison measures the complete fine-tuning procedure. It does not isolate FP_WEIGHT causally; a neutral-weight ablation would be needed to prove that the weighting itself caused the difference.

Best epoch row:
epoch         37.000000
phase          1.000000
train_loss     0.021573
val_loss       0.026091
iou            0.958820
precision      0.988918
recall         0.969235


In [30]:
threshold_candidates = [
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
]

threshold_rows = []

for threshold in threshold_candidates:
    metrics = evaluate_finetune(
        mobilenet_model,
        mobilenet_val_loader,
        threshold=threshold,
    )

    threshold_rows.append(
        {
            "threshold": threshold,
            "iou": metrics["iou"],
            "dice": metrics["dice"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "false_positive_pixels": metrics[
                "false_positive_pixels"
            ],
            "false_negative_pixels": metrics[
                "false_negative_pixels"
            ],
        }
    )

threshold_df = pd.DataFrame(
    threshold_rows
)

threshold_path = (
    MOBILENET_OUTPUT_DIR
    / "threshold_sweep.csv"
)

threshold_df.to_csv(
    threshold_path,
    index=False,
)

print(
    threshold_df.to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.4f}"
        ),
    )
)

print(
    "\nThreshold sweep saved to:",
    threshold_path,
)

print(
    "\nUse threshold 0.50 for the clean loss "
    "comparison. Raising the deployment threshold "
    "is a separate precision-versus-recall adjustment."
)

Validation:   0%|          | 0/6 [00:00<?, ?it/s]

Validation:   0%|          | 0/6 [00:00<?, ?it/s]

Validation:   0%|          | 0/6 [00:00<?, ?it/s]

Validation:   0%|          | 0/6 [00:00<?, ?it/s]

Validation:   0%|          | 0/6 [00:00<?, ?it/s]

 threshold    iou   dice  precision  recall  false_positive_pixels  false_negative_pixels
    0.4000 0.9596 0.9794     0.9872  0.9717                   9426                  21114
    0.5000 0.9588 0.9790     0.9889  0.9692                   8106                  22960
    0.6000 0.9574 0.9783     0.9904  0.9664                   6997                  25059
    0.7000 0.9554 0.9772     0.9919  0.9629                   5889                  27661
    0.8000 0.9518 0.9753     0.9935  0.9578                   4650                  31529

Threshold sweep saved to: /home/dragoon/peppermint/artifacts/finetune_hand_annotated_mobilenet_clean_5/threshold_sweep.csv

Use threshold 0.50 for the clean loss comparison. Raising the deployment threshold is a separate precision-versus-recall adjustment.
